In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))  # This brings 'src' into the path

import yaml

config = yaml.safe_load(open("../config.yaml", "r"))

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "8"
import torch
import torch.nn.functional as F

from src.model.models_fno_1 import FNO_1
from src.dataloader.dataloader_3d import dataset_sr
import numpy as np
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import random
import time

In [ ]:
import torch
state_dict = torch.load("../model/cfno_1/m8_nc32_res3_op2_acFalse_2/fno_1_weights_2025-07-01.pt")
for key, value in state_dict.items():
    print(f"{key}: {value.shape}")

In [ ]:
model = FNO_1(5, 32, 3, 2, 8, False)
model.load_state_dict(
    torch.load("../model/cfno_1/m8_nc32_res3_op2_acFalse_2/fno_1_weights_2025-07-01.pt", weights_only=True)
)

In [ ]:
dataset = dataset_sr()
loss = torch.nn.MSELoss()


In [ ]:
i = random.sample(range(40000), 1)[0]

scale_factor = 4
sr_factor = 4
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

# plot cuts at this z level
z_level = random.sample(range(128), 1)[0]
z_level_reduced = z_level // scale_factor
z_level_sr = int(z_level_reduced * sr_factor)

for ax in axes.flat:
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)

hr_state, lr_state_tensor, _, _ = dataset[i]

with torch.no_grad():
    start = time.time()
    sr_state = torch.squeeze(model(torch.unsqueeze(lr_state_tensor, 0), sr_factor), 0)
    print(time.time()-start)
#print(loss(sr_state, hr_state))
sr_state = sr_state.cpu().detach().numpy()
hr_state, lr_state = hr_state.numpy(), lr_state_tensor.numpy()
figures = []
figures.append(
    axes[0, 0].imshow(
        lr_state[0, :, :, z_level_reduced].T,
        origin="lower",
        extent=[0, 1, 0, 1],
        cmap="jet",
    )
)
axes[0, 0].set_ylabel(f"{i}")
figures.append(
    axes[0, 1].imshow(
        np.sqrt(
            lr_state[1, :, :, z_level_reduced] ** 2
            + lr_state[2, :, :, z_level_reduced] ** 2
        ).T,
        origin="lower",
        extent=[0, 1, 0, 1],
        cmap="jet",
    )
)
figures.append(
    axes[0, 2].imshow(
        lr_state[4, :, :, z_level_reduced].T,
        origin="lower",
        extent=[0, 1, 0, 1],
        cmap="jet",
    )
)

figures.append(
    axes[1, 0].imshow(
        hr_state[0, :, :, z_level].T, origin="lower", extent=[0, 1, 0, 1], cmap="jet"
    )
)
figures.append(
    axes[1, 1].imshow(
        np.sqrt(hr_state[1, :, :, z_level] ** 2 + hr_state[2, :, :, z_level] ** 2).T,
        origin="lower",
        extent=[0, 1, 0, 1],
        cmap="jet",
    )
)
figures.append(
    axes[1, 2].imshow(
        hr_state[4, :, :, z_level].T, origin="lower", extent=[0, 1, 0, 1], cmap="jet"
    )
)

figures.append(
    axes[2, 0].imshow(
        sr_state[0, :, :, z_level_sr].T, origin="lower", extent=[0, 1, 0, 1], cmap="jet"
    )
)
figures.append(
    axes[2, 1].imshow(
        np.sqrt(
            sr_state[1, :, :, z_level_sr] ** 2
            + sr_state[2, :, :, z_level_sr] ** 2
            + sr_state[3, :, :, z_level_sr] ** 2
        ).T,
        origin="lower",
        extent=[0, 1, 0, 1],
        cmap="jet",
    )
)
figures.append(
    axes[2, 2].imshow(
        sr_state[4, :, :, z_level_sr].T, origin="lower", extent=[0, 1, 0, 1], cmap="jet"
    )
)

for figs in figures:
    fig.colorbar(figs)
axes[0, 0].set_title("Density")
axes[0, 1].set_title("Velocity")
axes[0, 2].set_title("Pressure")

plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

substracted_state = sr_state - hr_state
figures = []
figures.append(
    axes[0].imshow(
        substracted_state[0, :, :, z_level].T,
        origin="lower",
        extent=[0, 1, 0, 1],
        cmap="seismic",
        vmin=-np.max(substracted_state[0, :, :, z_level]),
    )
)

figures.append(
    axes[1].imshow(
        np.sqrt(
            substracted_state[1, :, :, z_level_sr] ** 2
            + substracted_state[2, :, :, z_level_sr] ** 2
            + substracted_state[3, :, :, z_level_sr] ** 2
        ).T,
        origin="lower",
        extent=[0, 1, 0, 1],
        cmap="seismic",
        vmin=-np.max(
            np.sqrt(
                substracted_state[1, :, :, z_level_sr] ** 2
                + substracted_state[2, :, :, z_level_sr] ** 2
                + substracted_state[3, :, :, z_level_sr] ** 2
            )
        ),
    )
)

figures.append(
    axes[2].imshow(
        substracted_state[4, :, :, z_level].T,
        origin="lower",
        extent=[0, 1, 0, 1],
        cmap="seismic",
        vmin=-np.max(substracted_state[4, :, :, z_level]),
    )
)

for figs in figures:
    fig.colorbar(figs)

axes[0].set_title("Density")
axes[1].set_title("Velocity")
axes[2].set_title("Pressure")


In [ ]:
fig, axes = plt.subplots(1,3, figsize = (15, 5))

substracted_state = sr_state - hr_state
figures = []
figures.append(
    axes[0].imshow(
        substracted_state[0, :, :, z_level].T,
        origin="lower",
        extent=[0, 1, 0, 1],
        cmap="seismic",
        vmin=-np.max(substracted_state[0, :, :, z_level]),
    )
)

figures.append(
    axes[1].imshow(
        np.sqrt(
            substracted_state[1, :, :, z_level_sr] ** 2
            + substracted_state[2, :, :, z_level_sr] ** 2
            + substracted_state[3, :, :, z_level_sr] ** 2
        ).T,
        origin="lower",
        extent=[0, 1, 0, 1],
        cmap="seismic",
        vmin=-np.max(
            np.sqrt(
                substracted_state[1, :, :, z_level_sr] ** 2
                + substracted_state[2, :, :, z_level_sr] ** 2
                + substracted_state[3, :, :, z_level_sr] ** 2
            )),
    )
)

figures.append(
    axes[2].imshow(
        substracted_state[4, :, :, z_level].T,
        origin="lower",
        extent=[0, 1, 0, 1],
        cmap="seismic",
        vmin=-np.max(substracted_state[4, :, :, z_level]),
    )
)

for figs in figures:
    fig.colorbar(figs)

axes[0].set_title("Density")
axes[1].set_title("Velocity")
axes[2].set_title("Pressure")


In [ ]:
import src.utils.energy_mass as physics
import jax.numpy as jnp


In [ ]:
config, helper_data, registered_variables = physics.initialize_config()
sr_state = jnp.array(sr_state)
hr_state = jnp.array(hr_state)

In [ ]:
sr_ie_spectrum = physics.radial_spectrum(physics.internal_energy_spectrum(sr_state, 5/3, config, registered_variables))
sr_ke_spectrum = physics.radial_spectrum(physics.kinetic_energy_spectrum(sr_state, helper_data, config, registered_variables))
sr_mass_spectrum = physics.radial_spectrum(physics.mass_spectrum(sr_state, helper_data, config))

hr_ie_spectrum = physics.radial_spectrum(physics.internal_energy_spectrum(
    hr_state, 5 / 3, config, registered_variables
))
hr_ke_spectrum = physics.radial_spectrum(physics.kinetic_energy_spectrum(
    hr_state, helper_data, config, registered_variables
))
hr_mass_spectrum = physics.radial_spectrum(physics.mass_spectrum(hr_state, helper_data, config))


In [ ]:
config, helper_data, registered_variables = physics.initialize_config(state_shape=(5,32,32,32))
lr_state = jnp.array(lr_state)
lr_ie_spectrum = physics.radial_spectrum(
    physics.internal_energy_spectrum(lr_state, 5 / 3, config, registered_variables)
)
lr_ke_spectrum = physics.radial_spectrum(
    physics.kinetic_energy_spectrum(lr_state, helper_data, config, registered_variables)
)
lr_mass_spectrum = physics.radial_spectrum(
    physics.mass_spectrum(lr_state, helper_data, config)
)


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

spectrums = [
    sr_ie_spectrum,
    sr_ke_spectrum,
    sr_mass_spectrum,
    hr_ie_spectrum,
    hr_ke_spectrum,
    hr_mass_spectrum,
    lr_ie_spectrum,
    lr_ke_spectrum,
    lr_mass_spectrum,
]
for spectra, ax in zip(spectrums, axes.flatten()):
    ax.scatter(range(0,len(spectra)), spectra)

for ax in axes.flatten():
    ax.set_yscale("log")
    ax.set_xscale("log")


In [ ]:
print(len(sr_ie_spectrum))